In [ ]:
# 🔒 KHÓA TRƯỚC 11.5GB VRAM TESLA T4 GPU ĐỂ GIỮ COLAB GPU ACTIVE 100% VÀ BOOST CLOCK DƯỚI 1S
import torch

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
VRAM_LOCK_PLACEHOLDER = None

if HAS_CUDA:
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    total_mem = torch.cuda.get_device_properties(0).total_memory
    allocated_mem = torch.cuda.memory_allocated(0)
    free_mem_gb = (total_mem - allocated_mem) / 1024**3
    lock_gb = max(0.1, free_mem_gb - 3.5)
    VRAM_LOCK_PLACEHOLDER = torch.zeros((int(lock_gb * 1024), 1024, 512), device=DEVICE, dtype=torch.float16)
    print(f"🔒 [VRAM LOCK] Đã khóa thành công {lock_gb:.2f}GB VRAM trên Tesla T4 GPU!")
    print(f"⚡ Colab GPU RAM status: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB / {total_mem/1024**3:.2f} GB")
else:
    print("⚠️ Không tìm thấy GPU CUDA.")

🔒 [VRAM LOCK] Đã khóa thành công 11.06GB VRAM trên Tesla T4 GPU!
⚡ Colab GPU RAM status: 11.06 GB / 14.56 GB


In [ ]:
!nvidia-smi
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets huggingface_hub

Fri Aug  7 16:55:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 🚀 TOÀN BỘ TIẾN TRÌNH HUẤN LUYỆN GRPO MÔ HÌNH QWEN 2.5 CODER 0.5B (FP16 TENSOR CORES < 90 GIÂY)
import os, sys, re, json, torch
from huggingface_hub import login, HfApi

# Giải phóng VRAM lock placeholder từ Cell 1 nếu có
if "VRAM_LOCK_PLACEHOLDER" in globals():
    del globals()["VRAM_LOCK_PLACEHOLDER"]
torch.cuda.empty_cache()
torch.cuda.synchronize()

from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig

# 1. Đăng nhập HuggingFace Hub
HF_TOKEN = ""
login(token=HF_TOKEN)

# 2. Cấu hình mô hình chuyên logic Coder & GPU Target
BASE_MODEL = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
MODEL_REPO = "hoduyquocbao/xiangqi-r1-0.5b"
DATASET_REPO = "hoduyquocbao/xiangqi-r1-dataset"

print(f"⚡ [GPU TENSOR CORES FP16] Đang nạp {BASE_MODEL} lên GPU CUDA...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=1024,
    load_in_4bit=True,
    fast_inference=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("✅ Nạp Unsloth FP16 Tensor Cores cho Qwen 2.5 Coder 0.5B thành công!")

# 3. Máy chấm điểm GRPO (Reward Functions)
def format_reward_func(prompts, completions, **kwargs):
    rewards = []
    pattern = re.compile(r"^<thought>\n.*?\n</thought>\n[a-i][0-9][a-i][0-9]$", re.DOTALL)
    for completion in completions:
        text = completion.strip()
        if pattern.match(text): rewards.append(1.0)
        elif "<thought>" in text and "</thought>" in text: rewards.append(0.5)
        else: rewards.append(-1.0)
    return rewards

def rule_reward_func(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        text = completion.strip()
        match = re.search(r"([a-i][0-9][a-i][0-9])$", text)
        if not match: rewards.append(-5.0); continue
        move = match.group(1)
        if len(move) == 4 and move[0] in "abcdefghi" and move[2] in "abcdefghi": rewards.append(2.0)
        else: rewards.append(-5.0)
    return rewards

def quality_reward_func(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        text = completion.strip()
        match = re.search(r"([a-i][0-9][a-i][0-9])$", text)
        if not match: rewards.append(0.0); continue
        move = match.group(1)
        if move in ["b2e2", "h2e2", "b9c7", "h9g7", "c3c4", "g3g4"]: rewards.append(3.0)
        else: rewards.append(0.5)
    return rewards

# 4. Tải dataset và khởi chạy GPU GRPOTrainer Tốc Độ Siêu Tốc
print(f"📥 Đang nạp dataset cờ 3-in-1 từ HuggingFace Hub: {DATASET_REPO}...")
dataset = load_dataset(DATASET_REPO, split="train")

training_args = GRPOConfig(
    output_dir="outputs/xiangqi-r1-0.5b",
    learning_rate=1e-5,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_steps=5,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_generations=2,
    max_prompt_length=512,
    max_completion_length=128,
    max_steps=50,
    save_steps=25,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward_func, rule_reward_func, quality_reward_func],
    args=training_args,
    train_dataset=dataset,
)

print("============================================================")
print("🚀 BẮT ĐẦU HUẤN LUYỆN GRPO QWEN 2.5 CODER 0.5B TRÊN GPU CUDA")
print("============================================================")
trainer.train()

print(f"📤 Đang đẩy mô hình Qwen 2.5 Coder 0.5B lên HuggingFace Model Hub ({MODEL_REPO})...")
model.push_to_hub_merged(MODEL_REPO, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
print(f"✅ HOÀN TẤT ĐĂNG TẢI XIANGQI-R1 QWEN 2.5 CODER 0.5B LÊN HUB: https://huggingface.co/{MODEL_REPO}")

⚡ [GPU TENSOR CORES FP16] Đang nạp Qwen/Qwen2.5-Coder-0.5B-Instruct lên GPU CUDA...
==((====))==  Unsloth 2026.8.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅ Nạp Unsloth FP16 Tensor Cores cho Qwen 2.5 Coder 0.5B thành công!
📥 Đang nạp dataset cờ 3-in-1 từ HuggingFace Hub: hoduyquocbao/xiangqi-r1-dataset...


README.md:   0%|          | 0.00/1.51k [00:00<?, ?B/s]

train.json:   0%|          | 0.00/10.0M [00:00<?, ?B/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...r1-0.5b/model.safetensors:   3%|3         | 31.9MB /  988MB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:27<00:00, 27.70s/it]


Unsloth: Merge process complete. Saved to `/content/xiangqi-rim/hoduyquocbao/xiangqi-r1-0.5b`
✅ HOÀN TẤT ĐĂNG TẢI XIANGQI-R1 QWEN 2.5 CODER 0.5B LÊN HUB: https://huggingface.co/hoduyquocbao/xiangqi-r1-0.5b


In [36]:
# 🚀 HUẤN LUYỆN NÂNG CAO THẾ HỆ TIẾP THEO (QWEN 2.5 CODER 0.5B CONTINUAL GRPO 200 STEPS) TRỰC TIẾP TRÊN GPU TESLA T4
import os, sys, re, json, torch
from huggingface_hub import login
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig

HF_TOKEN = ""
login(token=HF_TOKEN)

MODEL_REPO = "hoduyquocbao/xiangqi-r1-0.5b"
DATASET_REPO = "hoduyquocbao/xiangqi-r1-dataset"

print(f"⚡ [GPU CONTINUAL TRAINING] Đang nạp checkpoint 16-bit merged từ Hub: {MODEL_REPO}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_REPO,
    max_seq_length=1024,
    load_in_4bit=True,
    fast_inference=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def format_reward_func(prompts, completions, **kwargs):
    rewards = []
    pattern = re.compile(r"^<thought>\n.*?\n</thought>\n[a-i][0-9][a-i][0-9]$", re.DOTALL)
    for completion in completions:
        text = completion.strip()
        if pattern.match(text): rewards.append(1.0)
        elif "<thought>" in text and "</thought>" in text: rewards.append(0.5)
        else: rewards.append(-1.0)
    return rewards

def rule_reward_func(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        text = completion.strip()
        match = re.search(r"([a-i][0-9][a-i][0-9])$", text)
        if not match: rewards.append(-5.0); continue
        move = match.group(1)
        if len(move) == 4 and move[0] in "abcdefghi" and move[2] in "abcdefghi": rewards.append(2.0)
        else: rewards.append(-5.0)
    return rewards

def quality_reward_func(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        text = completion.strip()
        match = re.search(r"([a-i][0-9][a-i][0-9])$", text)
        if not match: rewards.append(0.0); continue
        move = match.group(1)
        if move in ["b2e2", "h2e2", "b9c7", "h9g7", "c3c4", "g3g4"]: rewards.append(3.0)
        else: rewards.append(0.5)
    return rewards

dataset = load_dataset(DATASET_REPO, split="train")

training_args = GRPOConfig(
    output_dir="outputs/xiangqi-r1-0.5b-continual",
    learning_rate=2e-5,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_generations=4,
    max_prompt_length=512,
    max_completion_length=128,
    max_steps=200,
    save_steps=50,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward_func, rule_reward_func, quality_reward_func],
    args=training_args,
    train_dataset=dataset,
)

print("============================================================")
print("🚀 BẮT ĐẦU HUẤN LUYỆN CONTINUAL GRPO 200 STEPS TRÊN GPU CUDA")
print("============================================================")
trainer.train()

model.push_to_hub_merged(MODEL_REPO, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
print(f"✅ ĐÃ CẬP NHẬT CONTINUAL LEARNING 200 STEPS LÊN HUB: https://huggingface.co/{MODEL_REPO}")


⚡ [GPU CONTINUAL TRAINING] Đang nạp checkpoint 16-bit merged từ Hub: hoduyquocbao/xiangqi-r1-0.5b...
==((====))==  Unsloth 2026.8.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Unsloth: Switching to float32 training since model cannot work with float16
🚀 BẮT ĐẦU HUẤN LUYỆN CONTINUAL GRPO 200 STEPS TRÊN GPU CUDA


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,000 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Unsloth: Will smartly offload gradients to save VRAM!


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...r1-0.5b/model.safetensors:   2%|2         | 24.0MB /  988MB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:36<00:00, 36.82s/it]


Unsloth: Merge process complete. Saved to `/content/xiangqi-rim/hoduyquocbao/xiangqi-r1-0.5b`
✅ ĐÃ CẬP NHẬT CONTINUAL LEARNING 200 STEPS LÊN HUB: https://huggingface.co/hoduyquocbao/xiangqi-r1-0.5b


In [37]:
# 🚀 HUẤN LUYỆN NÂNG CAO TĂNG CƯỜNG ĐỢT 3 (QWEN 2.5 CODER 0.5B CONTINUAL GRPO 300 STEPS - DEBT PURSUIT)
import os, sys, re, json, torch
from huggingface_hub import login
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig

HF_TOKEN = ""
login(token=HF_TOKEN)

MODEL_REPO = "hoduyquocbao/xiangqi-r1-0.5b"
DATASET_REPO = "hoduyquocbao/xiangqi-r1-dataset"

print(f"⚡ [GPU BATCH 3] Đang nạp checkpoint 16-bit merged mới nhất từ Hub: {MODEL_REPO}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_REPO,
    max_seq_length=1024,
    load_in_4bit=True,
    fast_inference=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def format_reward_func(prompts, completions, **kwargs):
    rewards = []
    pattern = re.compile(r"^<thought>\n.*?\n</thought>\n[a-i][0-9][a-i][0-9]$", re.DOTALL)
    for completion in completions:
        text = completion.strip()
        if pattern.match(text): rewards.append(1.5)
        elif "<thought>" in text and "</thought>" in text: rewards.append(0.8)
        else: rewards.append(-2.0)
    return rewards

def rule_reward_func(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        text = completion.strip()
        match = re.search(r"([a-i][0-9][a-i][0-9])$", text)
        if not match: rewards.append(-5.0); continue
        move = match.group(1)
        if len(move) == 4 and move[0] in "abcdefghi" and move[2] in "abcdefghi": rewards.append(3.0)
        else: rewards.append(-5.0)
    return rewards

def quality_reward_func(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        text = completion.strip()
        match = re.search(r"([a-i][0-9][a-i][0-9])$", text)
        if not match: rewards.append(0.0); continue
        move = match.group(1)
        if move in ["b2e2", "h2e2", "b9c7", "h9g7", "c3c4", "g3g4", "i0h0", "a0b0"]: rewards.append(4.0)
        else: rewards.append(1.0)
    return rewards

dataset = load_dataset(DATASET_REPO, split="train")

training_args = GRPOConfig(
    output_dir="outputs/xiangqi-r1-0.5b-batch3",
    learning_rate=3e-5,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_generations=4,
    max_prompt_length=512,
    max_completion_length=128,
    max_steps=300,
    save_steps=50,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward_func, rule_reward_func, quality_reward_func],
    args=training_args,
    train_dataset=dataset,
)

print("============================================================")
print("🚀 BẮT ĐẦU HUẤN LUYỆN BATCH 3 GRPO 300 STEPS TRÊN GPU CUDA")
print("============================================================")
trainer.train()

model.push_to_hub_merged(MODEL_REPO, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
print(f"✅ ĐÃ ĐẨY BATCH 3 CONTINUAL LEARNING 300 STEPS LÊN HUB: https://huggingface.co/{MODEL_REPO}")


⚡ [GPU BATCH 3] Đang nạp checkpoint 16-bit merged mới nhất từ Hub: hoduyquocbao/xiangqi-r1-0.5b...
==((====))==  Unsloth 2026.8.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Unsloth: Switching to float32 training since model cannot work with float16
🚀 BẮT ĐẦU HUẤN LUYỆN BATCH 3 GRPO 300 STEPS TRÊN GPU CUDA


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,000 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Unsloth: Will smartly offload gradients to save VRAM!


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...r1-0.5b/model.safetensors:   1%|          | 7.97MB /  988MB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:36<00:00, 36.98s/it]


Unsloth: Merge process complete. Saved to `/content/xiangqi-rim/hoduyquocbao/xiangqi-r1-0.5b`
✅ ĐÃ ĐẨY BATCH 3 CONTINUAL LEARNING 300 STEPS LÊN HUB: https://huggingface.co/hoduyquocbao/xiangqi-r1-0.5b
